# Import libraries

In [1]:
import uuid
import copy
from datetime import datetime
from pathlib import Path
import getpass
import sys
import pandas as pd
import numpy as np
import json
import pickle
import re
import os
import glob
from typing import Dict, List, Optional, Any, Tuple
from tqdm.notebook import tqdm
import duckdb

sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.lcmsruns_tools as lrt
import metatlas2.ms1_ms2_analysis as msa

pd.options.display.max_colwidth = 300

# Define project variables

In [2]:
# Project setup
DATA_DIR = Path("/Users/BKieft/Metabolomics/metatlas2/data/")
DATABASES_DIR = DATA_DIR / "databases"
PROJECT = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
PROJECT_DIR = DATA_DIR / "test_data/projects" / PROJECT

# QC Files Configuration
QC_FILE_PATH = PROJECT_DIR / "lcmsruns"  # Base path to search for QC files
MIN_QC_FILES = 1  # Minimum number of QC files required for RT correction

# Database Configuration
DATABASE_PATH = "/Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/metatlas.duckdb"
PROJECT_DB_PATH = PROJECT_DIR / f"{PROJECT}.duckdb"  # Project-specific database

# Atlas Configuration
# QC (to make model)
QC_ATLAS_NAME = "HILIC Positive QC Default"  # Atlas name for QC compounds
QC_ATLAS_UID = "atlas-b8ae18bcb00549b7"  # Optional: specify atlas UID directly
# Target (to be aligned by model)
TARGET_ATLAS_NAME = "HILIC Positive ISTD Default"  # or
TARGET_ATLAS_UID = "atlas-62499fad4e104743"

# Mass Tolerance Settings
DEFAULT_MZ_TOLERANCE_PPM = 5.0  # Default m/z tolerance in ppm for compound matching

# Retention Time Settings
RT_WINDOW_EXPANSION = 1  # Extra RT window (minutes) around Atlas RT for QC peak search
MIN_PEAK_INTENSITY = 1e5  # Minimum peak intensity for QC peak consideration

# Polynomial Model Settings - ALWAYS USE SECOND-ORDER POLYNOMIAL
POLYNOMIAL_DEGREE = 2  # Fixed second-order polynomial (quadratic) for RT correction
MIN_OBSERVATIONS_PER_COMPOUND = 1 # Minimum observations (files) per compound to be included in RT correction model
MIN_COMPOUNDS_FOR_MODELING = 2  # Minimum compounds needed to build RT correction model
EXCLUDE_INCHIKEYS = ["OVRNDRQMDRJTHS-ZEUBEQSHSA-N"] # Optional: compounds from QC to exclude from RT alignment model

# Model Selection Criteria
R2_THRESHOLD = 0.7  # Minimum R-squared for acceptable model performance
MAX_RESIDUAL_RT = 1  # Maximum acceptable residual RT difference (minutes)

# Output Configuration
TIMESTAMP = datetime.now().isoformat()  # Timestamp for output files
GENERATE_PLOTS = True  # Generate diagnostic plots
SAVE_INTERMEDIATE_DATA = True  # Save intermediate data for debugging
ANALYST_NAME = getpass.getuser()
METADATA = {
    "timestamp": TIMESTAMP,
    "analyst": ANALYST_NAME
}

# Create Project Database and Load LCMS Runs

In [3]:
# Create project-specific database
dbi.create_project_database(PROJECT_DB_PATH)

# Save LCMS runs to project database
lcmsrun_files = dbi.save_lcmsruns_to_db(
    project_db_path=PROJECT_DB_PATH,
    project_path=PROJECT_DIR,
    project_metadata=METADATA
)

print(f"LCMS run files summary:")
for analysis_type, files in lcmsrun_files.items():
    print(f"  {analysis_type}: {len(files)} files")

Project database created at /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb
20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5
20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_36_Fall-VFa_7_Rg70to1050-CE102040norm-veg-core-S1_Run136.h5
20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_9_Spring-RSp_6_Rg70to1050-CE102040norm-rhizo-S1_Run99.h5
20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_28_Spring-VSp_2_Rg70to1050-CE102040norm-veg-core-ISTD_Run31.h5
20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_48_Fall-BFa_7_Rg70to1050-CE102040norm-nonveg-core-S1_Run127.h5
20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_54_ExCtrl_6_Rg70to1050-CE102040norm-

# Load QC Atlas from Database

In [ ]:
# Load QC compounds from main database using new atlas system
qc_compounds_df, CHROMATOGRAPHY, POLARITY = dbi.get_atlas_compounds_from_db(
    db_path=DATABASE_PATH,
    atlas_name=QC_ATLAS_NAME,
    atlas_uid=QC_ATLAS_UID
)

# Filter out compounds without RT or MZ data
valid_qc_compounds_df = qc_compounds_df.dropna(subset=['rt_peak', 'mz'])

# Check load
if len(valid_qc_compounds_df) == 0:
    print("No valid compounds found in QC Atlas. Exiting.")
    exit(1)

print(f"Inferred method parameters from QC atlas:")
print(f"  Chromatography: {CHROMATOGRAPHY}")
print(f"  Polarity: {POLARITY}")

display(pd.DataFrame({
    'Atlas Name': [QC_ATLAS_NAME],
    'Atlas UID': [QC_ATLAS_UID],
    'Compounds': [len(valid_qc_compounds_df)],
    'Polarity': [POLARITY],
    'Chromatography': [CHROMATOGRAPHY]
}))

Retrieved 20 compounds for atlas HILIC Positive QC Default (atlas-b8ae18bcb00549b7)


,Atlas Name,Atlas UID,Compounds,Polarity,Chromatography
0,HILIC Positive QC Default,atlas-b8ae18bcb00549b7,20,positive,HILIC


In [ ]:
# Load target compounds from main database using new atlas system  
target_compounds_df, target_chromatography, target_polarity = dbi.get_atlas_compounds_from_db(
    db_path=DATABASE_PATH,
    atlas_name=TARGET_ATLAS_NAME,
    atlas_uid=TARGET_ATLAS_UID
)

# Verify target atlas matches QC atlas method parameters
if target_chromatography != CHROMATOGRAPHY or target_polarity != POLARITY:
    print(f"Warning: Target atlas method ({target_chromatography}/{target_polarity}) differs from QC atlas ({CHROMATOGRAPHY}/{POLARITY})")
    print("Using QC atlas parameters for consistency")

# Filter out compounds without RT or MZ data
valid_target_compounds_df = target_compounds_df.dropna(subset=['rt_peak', 'mz'])

# Check load
if len(valid_target_compounds_df) == 0:
    print("No valid compounds found in Target Atlas. Exiting.")
    exit(1)

display(pd.DataFrame({
    'Atlas Name': [TARGET_ATLAS_NAME],
    'Atlas UID': [TARGET_ATLAS_UID],
    'Compounds': [len(valid_target_compounds_df)],
    'Polarity': [target_polarity],
    'Chromatography': [target_chromatography]
}))

Retrieved 65 compounds for atlas HILIC Positive ISTD Default (atlas-62499fad4e104743)


,Atlas Name,Atlas UID,Compounds,Polarity,Chromatography
0,HILIC Positive ISTD Default,atlas-62499fad4e104743,65,positive,HILIC


# Load LCMS run files from database

In [ ]:
# Get QC files using inferred chromatography and polarity from atlas
qc_files = dbi.get_valid_qc_files(
    project_db_path=PROJECT_DB_PATH, 
    chromatography=CHROMATOGRAPHY, 
    polarity=POLARITY, 
    min_qc_files=MIN_QC_FILES
)

Valid QC files ready for processing: 2
  20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5
  20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_33_Summer-VSu_6_Rg70to1050-CE102040norm-veg-core-QC_Run107.h5


# Extract RT Data from QC files

In [22]:
# Extract MS1 data from all QC files
print("Extracting targeted MS1 data from QC files...")

# Extract MS1 data from each QC file
all_ms1_data = []
extraction_stats = {
    'total_files': len(qc_files),
    'successful_extractions': 0,
    'failed_extractions': 0,
    'total_peaks': 0
}

for qc_file in tqdm(qc_files, desc="Extracting MS1 data"):
    ms1_data = msa.extract_ms1_data_from_file(qc_file, POLARITY)
    
    if len(ms1_data) > 0:
        all_ms1_data.append(ms1_data)
        extraction_stats['successful_extractions'] += 1
        extraction_stats['total_peaks'] += len(ms1_data)
    else:
        extraction_stats['failed_extractions'] += 1

# Combine all MS1 data
if all_ms1_data:
    combined_ms1_data = pd.concat(all_ms1_data, ignore_index=True)
    print(f"\nMS1 data extraction completed")
    print(f"  Successful file extractions: {extraction_stats['successful_extractions']}/{extraction_stats['total_files']}")
    print(f"  Total MS1 peaks: {extraction_stats['total_peaks']:,}")
    
    # Display data summary
    print(f"\nMS1 Data Summary:")
    print(f"  RT range: {combined_ms1_data['rt'].min():.2f} - {combined_ms1_data['rt'].max():.2f} min")
    print(f"  m/z range: {combined_ms1_data['mz'].min():.4f} - {combined_ms1_data['mz'].max():.4f}")
    print(f"  Intensity range: {combined_ms1_data['i'].min():.0f} - {combined_ms1_data['i'].max():.0f}")
    
else:
    raise ValueError("No MS1 data could be extracted from any QC files")

Extracting targeted MS1 data from QC files...


Extracting MS1 data:   0%|          | 0/2 [00:00<?, ?it/s]

Extracted 44766 MS1 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5
Extracted 42487 MS1 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_33_Summer-VSu_6_Rg70to1050-CE102040norm-veg-core-QC_Run107.h5

MS1 data extraction completed
  Successful file extractions: 2/2
  Total MS1 peaks: 87,253

MS1 Data Summary:
  RT range: 0.00 - 20.97 min
  m/z range: 70.1983 - 1049.5833
  Intensity range: 100001 - 406734272


# Match QC Atlas Compounds to Extracted QC MS1 Data

In [23]:
matches_df, matching_stats = find_qc_compounds_in_peaks(valid_qc_compounds_df, combined_ms1_data, DEFAULT_MZ_TOLERANCE_PPM, RT_WINDOW_EXPANSION)

Matching QC Atlas compounds with QC peak data


Matching QC compounds:   0%|          | 0/20 [00:00<?, ?it/s]

Compounds with matches: 20
Compounds without matches: 0
Total peak matches: 747
Mean m/z error: -0.21 ± 0.51 ppm
Mean RT difference: 0.028 ± 0.119 min


# Build RT Alignment Model

In [ ]:
if EXCLUDE_INCHIKEYS:
    # Ensure 'inchi_key' column exists in matches_df
    if 'inchi_key' not in matches_df.columns:
        raise ValueError("matches_df must contain an 'inchi_key' column to filter by InChI Key.")
    before_count = len(matches_df)
    matches_df = matches_df[~matches_df['inchi_key'].isin(EXCLUDE_INCHIKEYS)]
    after_count = len(matches_df)
    print(f"Filtered out {before_count - after_count} matches by InChI Key.")

# Build RT correction model using QC compound statistics

# Aggregate matches to calculate median/mean observed RT values for each compound
compound_rt_stats = matches_df.groupby(['compound_uid', 'compound_name', 'inchi_key']).agg({
    'atlas_rt_peak': 'first',
    'atlas_rt_min': 'first',
    'atlas_rt_max': 'first',
    'atlas_mz': 'first',
    'observed_rt': ['mean', 'median', 'std', 'count'],
    'observed_mz': ['mean', 'std'],
    'observed_intensity': ['mean', 'median', 'max'],
    'rt_difference': ['mean', 'median', 'std'],
    'mz_error_ppm': ['mean', 'std']
}).round(4)

# Flatten column names
compound_rt_stats.columns = [
    'ref_rt_peak',           # Atlas RT peak (reference)
    'ref_rt_min',            # Atlas RT min
    'ref_rt_max',            # Atlas RT max  
    'ref_mz',                # Atlas m/z
    'exp_rt_mean',           # Mean observed RT across files
    'exp_rt_median',         # Median observed RT across files (more robust)
    'exp_rt_std',            # Standard deviation of observed RT
    'observation_count',     # Number of files where compound was observed
    'exp_mz_mean',           # Mean observed m/z
    'exp_mz_std',            # Std dev of observed m/z
    'exp_intensity_mean',    # Mean intensity
    'exp_intensity_median',  # Median intensity  
    'exp_intensity_max',     # Max intensity
    'rt_diff_mean',          # Mean RT difference (observed - atlas)
    'rt_diff_median',        # Median RT difference
    'rt_diff_std',           # Std dev of RT difference
    'mz_error_mean',         # Mean m/z error (ppm)
    'mz_error_std'           # Std dev of m/z error (ppm)
]

# Reset index to get compound info as columns
compound_rt_stats = compound_rt_stats.reset_index()

# Display summary statistics
print(f"\nRT Statistics Summary:")
print(f"  Atlas RT range: {compound_rt_stats['ref_rt_peak'].min():.2f} - {compound_rt_stats['ref_rt_peak'].max():.2f} min")
print(f"  Observed RT range (median): {compound_rt_stats['exp_rt_median'].min():.2f} - {compound_rt_stats['exp_rt_median'].max():.2f} min")
print(f"  Mean RT difference (observed - atlas): {compound_rt_stats['rt_diff_median'].mean():.3f} ± {compound_rt_stats['rt_diff_median'].std():.3f} min")

# Show the compound stats table
print(f"\nCompound RT Statistics:")
display(compound_rt_stats[['compound_name', 'inchi_key', 'ref_rt_peak', 'exp_rt_median', 'rt_diff_median', 
                          'observation_count', 'exp_rt_std']])

# Prepare modeling data from compound RT statistics
modeling_data = []

# Filter compounds with sufficient observations for reliable modeling
reliable_compounds = compound_rt_stats[compound_rt_stats['observation_count'] >= MIN_OBSERVATIONS_PER_COMPOUND]
print(f"Using {len(reliable_compounds)} compounds with ≥{MIN_OBSERVATIONS_PER_COMPOUND} observations (QC files) for modeling")
if len(reliable_compounds) < MIN_COMPOUNDS_FOR_MODELING:
    raise ValueError(f"Insufficient compounds for modeling. Need at least {MIN_COMPOUNDS_FOR_MODELING}, but found {len(reliable_compounds)}")

# Extract X (Atlas RT) and y (observed median RT) for modeling
X_atlas_rt = reliable_compounds['ref_rt_peak'].values
y_observed_rt = reliable_compounds['exp_rt_median'].values

# Create modeling dataset
modeling_results_df = reliable_compounds.copy()

# Build second-order polynomial model (as specified in config)
print(f"Building polynomial model (degree {POLYNOMIAL_DEGREE})...")

best_model = build_polynomial_model(X_atlas_rt, y_observed_rt, POLYNOMIAL_DEGREE)

# Add predictions and residuals to modeling results
modeling_results_df['predicted_rt'] = best_model['y_pred']
modeling_results_df['residual'] = y_observed_rt - best_model['y_pred']
modeling_results_df['abs_residual'] = np.abs(modeling_results_df['residual'])

# Model evaluation
print(f"\nModel built successfully:")
print(f"  Model type: Polynomial degree {best_model['degree']}")
print(f"  R² = {best_model['r2']:.4f}")
print(f"  RMSE = {best_model['rmse']:.4f} min")
print(f"  Max residual = {modeling_results_df['abs_residual'].max():.4f} min")

# Display polynomial equation
equation = format_polynomial_equation(best_model)
print(f"  Equation: {equation}")
best_model['equation'] = equation

# Check model quality
if best_model['r2'] < R2_THRESHOLD:
    print(f"Warning: Model R² ({best_model['r2']:.4f}) is below threshold ({R2_THRESHOLD})")

if modeling_results_df['abs_residual'].max() > MAX_RESIDUAL_RT:
    print(f"Warning: Maximum residual ({modeling_results_df['abs_residual'].max():.4f} min) exceeds threshold ({MAX_RESIDUAL_RT} min)")

# Add compounds
best_model['compounds_used_for_modeling'] = reliable_compounds['compound_uid'].tolist()

# Store modeling data for later use
modeling_data = modeling_results_df.to_dict('records')

Filtered out 2 matches by InChI Key.

RT Statistics Summary:
  Atlas RT range: 1.09 - 17.01 min
  Observed RT range (median): 1.25 - 17.14 min
  Mean RT difference (observed - atlas): 0.016 ± 0.109 min

Compound RT Statistics:


,compound_name,inchi_key,ref_rt_peak,exp_rt_median,rt_diff_median,observation_count,exp_rt_std
0,"thymine (U - 13C, 15N)",RWQNBRDOKXIBIV-BNUYUSEDSA-N,1.2552,1.353900,0.0986,2,0.0003
1,"tryptophan (U - 13C, 15N)",QIVBCDIJIAJPQS-HNEHNLBWSA-N,10.1566,10.180700,0.0240,2,0.0109
2,"alanine (U - 13C, 15N)",QNAYBMKLOCPYGJ-UVYXLFMMSA-N,13.4051,13.358000,-0.0471,2,0.0095
3,"asparagine (U - 13C, 15N)",DCXYFEDJOCDNAF-FLEDYEEXSA-N,14.3681,14.318600,-0.0495,2,0.0003
4,"arginine (U - 13C, 15N)",ODKSFYDXXFIFQN-GZYYCROJSA-N,16.9399,17.056601,0.1167,2,0.0058
5,ABMBA (unlabeled),LCMZECCEEOQWLQ-UHFFFAOYSA-N,1.0938,1.253400,0.1596,2,0.0001
6,inosine (U - 15N),UGQMRVRMYYASKQ-SHKQESLVSA-N,5.4342,5.197600,-0.2366,2,0.0412
7,"aspartic acid (U - 13C, 15N)",CKLJMWTZIZZHCS-NYTNKSBQSA-N,16.1304,16.169500,0.0391,2,0.0116
8,"methionine (U - 13C, 15N)",FFEARJCKVFRZRR-XAFSXMPTSA-N,10.4410,10.409400,-0.0315,2,0.0075
9,guanine (U - 15N),UYTPUPDQBNUYGX-CIKZIQIKSA-N,6.2654,6.210600,-0.0548,2,0.0090


Using 19 compounds with ≥1 observations (QC files) for modeling
Building polynomial model (degree 2)...

Model built successfully:
  Model type: Polynomial degree 2
  R² = 0.9998
  RMSE = 0.0820 min
  Max residual = 0.2197 min
  Equation: RT_corrected = 0.188479 + 0.946110 * RT_atlas + 0.002824 * RT_atlas^2


# Create RT-Corrected Experimental Data and Save to Database

In [25]:
# Create RT-corrected experimental data and save to project database
print("Creating RT-corrected experimental data and saving to project database...")

# Save RT alignment model to database
rt_alignment_uid = dbi.save_rt_alignment_model_to_db(PROJECT_DB_PATH, best_model, qc_files, modeling_data)

# Create RT-corrected experimental data for target compounds
conn = duckdb.connect(str(PROJECT_DB_PATH))

# Clear existing experimental data
conn.execute("DELETE FROM mz_rt_experimental")

correction_stats = []

for _, compound in valid_target_compounds_df.iterrows():
    compound_uid = compound['compound_uid']
    original_rt_peak = compound['rt_peak']
    
    # Apply RT correction using the polynomial model
    rt_poly = best_model['poly_features'].transform([[original_rt_peak]])
    corrected_rt_peak = best_model['model'].predict(rt_poly)[0]
    rt_shift = corrected_rt_peak - original_rt_peak
    
    # Calculate corrected RT window
    rt_window_size = compound['rt_max'] - compound['rt_min']
    corrected_rt_min = corrected_rt_peak - (rt_window_size / 2)
    corrected_rt_max = corrected_rt_peak + (rt_window_size / 2)
    
    # Generate new experimental UID
    mz_rt_experimental_uid = f"mzrt-exp-{uuid.uuid4().hex[:32]}"
    
    # Insert into experimental data table
    conn.execute("""
        INSERT INTO mz_rt_experimental VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        mz_rt_experimental_uid,
        compound_uid,
        corrected_rt_peak,
        corrected_rt_min,
        corrected_rt_max,
        compound['mz'],
        compound['mz_tolerance'],
        compound['adduct'],
        True,  # rt_correction_applied
        rt_shift,
        compound['mz_rt_reference_uid'],
        ANALYST_NAME,
        TIMESTAMP
    ))
    
    correction_stats.append({
        'compound_name': compound['compound_name'],
        'compound_uid': compound_uid,
        'mz_rt_reference_uid': compound['mz_rt_reference_uid'],
        'mz_rt_experimental_uid': mz_rt_experimental_uid,
        'original_rt': original_rt_peak,
        'corrected_rt': corrected_rt_peak,
        'rt_shift': rt_shift
    })

conn.close()

# Create summary
correction_df = pd.DataFrame(correction_stats)
summary = {
    'total_compounds': len(valid_target_compounds_df),
    'corrected_compounds': len(correction_stats),
    'rt_alignment_uid': rt_alignment_uid,
    'mean_correction': correction_df['rt_shift'].mean() if not correction_df.empty else 0,
    'std_correction': correction_df['rt_shift'].std() if not correction_df.empty else 0,
    'min_correction': correction_df['rt_shift'].min() if not correction_df.empty else 0,
    'max_correction': correction_df['rt_shift'].max() if not correction_df.empty else 0
}

print(f"RT correction completed: {summary['corrected_compounds']}/{summary['total_compounds']} compounds corrected")
if summary['corrected_compounds']:
    print(f"Correction statistics: mean = {summary['mean_correction']:.4f}, std = {summary['std_correction']:.4f} min")

print(f"RT alignment model and experimental data saved to project database: {PROJECT_DB_PATH}")
print(f"RT alignment model UID: {rt_alignment_uid}")

# Display some corrected compounds
print(f"\nSample of corrected compounds:")
display(correction_df.head(10))

Creating RT-corrected experimental data and saving to project database...
RT alignment model saved to database with UID: rta-78caaf5bfec74cff
RT correction completed: 65/65 compounds corrected
Correction statistics: mean = 0.0025, std = 0.0657 min
RT alignment model and experimental data saved to project database: /Users/BKieft/Metabolomics/metatlas2/data/tests/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb
RT alignment model UID: rta-78caaf5bfec74cff

Sample of corrected compounds:


,compound_name,compound_uid,mz_rt_reference_uid,mz_rt_experimental_uid,original_rt,corrected_rt,rt_shift
0,ABMBA (unlabeled),comp-2119ab5f47724223,mzrt-ec2095d552c74113,mzrt-exp-381a86a445d24fae,1.093806,1.226719,0.132913
1,nicotinamide (U - 13C),comp-3d54804959dc4f14,mzrt-6c20df52cc7248b9,mzrt-exp-96ae354fc6104a29,1.146589,1.276991,0.130402
2,nicotinamide,comp-b8148872ded94c1d,mzrt-b9f57183196e4516,mzrt-exp-8bec17de261b457b,1.224396,1.351126,0.126730
3,"thymine (U - 13C, 15N)",comp-02742dd2da34468b,mzrt-81ec582ddd2743cb,mzrt-exp-cafc5ee9748b405f,1.255231,1.380515,0.125284
4,thymine,comp-4892937a14974a4f,mzrt-18f65a7e84c143a5,mzrt-exp-604303d9923b4766,1.257993,1.383148,0.125155
5,"uracil (U - 13C, 15N)",comp-7490cf2b00094bfd,mzrt-f0eb9398c4874e4b,mzrt-exp-b4a2c30cd7274283,1.393700,1.512557,0.118858
6,uracil,comp-f82662ac39c645d5,mzrt-ddcd7034030345ea,mzrt-exp-c6a00cf7da6d429e,1.396619,1.515342,0.118724
7,adenine (U - 15N),comp-c2474f67f12848a8,mzrt-aecb73b463d7415c,mzrt-exp-8545412607084123,2.677602,2.742031,0.064429
8,adenine,comp-44cb8cc848444b09,mzrt-37347e692ad24f7a,mzrt-exp-e00ca318dae14d61,2.675524,2.740033,0.064510
9,hypoxanthine (U - 15N),comp-736fbaf9394b4977,mzrt-2b0227f6a606480e,mzrt-exp-b3d9ff7a46b44a89,3.102967,3.151417,0.048449


In [27]:
db_path = PROJECT_DB_PATH
compound_uid = "comp-2119ab5f47724223"

conn = duckdb.connect(str(db_path))
df = conn.execute(
    "SELECT * FROM mz_rt_experimental WHERE compound_uid = ?", [compound_uid]
).df()
conn.close()

print(df)


      mz_rt_experimental_uid           compound_uid   rt_peak    rt_min  \
0  mzrt-exp-381a86a445d24fae  comp-2119ab5f47724223  1.226719  0.926719   

     rt_max        mz  mz_tolerance  adduct  rt_correction_applied  rt_shift  \
0  1.526719  229.9811           5.0  [M+H]+                   True  0.132913   

  source_mz_rt_reference_uid created_by              creation_time  
0      mzrt-ec2095d552c74113     BKieft 2025-08-22 09:12:45.817584  


In [ ]:
db_path = DATABASE_PATH
compound_uid = "comp-2119ab5f47724223"

conn = duckdb.connect(str(db_path))
df = conn.execute(
    "SELECT * FROM mz_rt_references WHERE compound_uid = ?", [compound_uid]
).df()
conn.close()

print(df)

     mz_rt_reference_uid           compound_uid   rt_peak    rt_min    rt_max  \
0  mzrt-e447f92c261b42ef  comp-2119ab5f47724223  1.121446  0.821446  1.421446   
1  mzrt-ec2095d552c74113  comp-2119ab5f47724223  1.093806  0.793806  1.393806   

          mz  mz_tolerance  adduct chromatography  polarity confidence  \
0  227.96656           5.0  [M-H]-         HILICZ  negative              
1  229.98110           5.0  [M+H]+         HILICZ  positive              

                source created_by              creation_time  
0  HILICZ_ISTD_NEG.tsv     BKieft 2025-08-21 12:50:45.244758  
1  HILICZ_ISTD_POS.tsv     BKieft 2025-08-21 12:51:01.401490  
